# VUS Drift — Phase 1

patient=Claude-Sonnet · bots = **all 6** (Claude sonnet+haiku, GPT gpt-5.4+mini, Gemini 3.5-flash+3.1-flash-lite), all non-reasoning. 10 turns. The random-bot run draws from all 6. If a model 404s, run cell 4b to list valid IDs and edit CFG['models']['bots'].

### 1. Install

In [ ]:
!pip install -q openai anthropic google-genai

### 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
FOLDER='/content/drive/MyDrive/VUS_1806'
os.makedirs(FOLDER,exist_ok=True); os.chdir(FOLDER); print('cwd:',os.getcwd())

### 3. Keys (Colab Secrets)

In [ ]:
# Colab Secrets (key panel) - v8 names
import os
try:
    from google.colab import userdata
    for k in ['GOOGLE_API_KEY','CHATGPT_API_KEY','OPENAI_API_KEY','ANTHROPIC_API_KEY']:
        try:
            val=userdata.get(k)
            if val: os.environ[k]=val; print('[ok]',k)
        except Exception: pass
except Exception: print('not in Colab')
print('present:',[k for k in ['GOOGLE_API_KEY','CHATGPT_API_KEY','OPENAI_API_KEY','ANTHROPIC_API_KEY'] if os.environ.get(k)])

### 4b. (optional) verify available model IDs

In [ ]:
# OPTIONAL: verify which model IDs your API exposes (run if you get a 404)
try:
    from google import genai
    g=genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
    print('GEMINI:',[m.name for m in g.models.list() if 'gemini' in m.name.lower()][:20])
except Exception as e: print('gemini list err:',e)
try:
    from openai import OpenAI
    o=OpenAI(api_key=os.environ.get('CHATGPT_API_KEY') or os.environ.get('OPENAI_API_KEY'))
    print('GPT:',sorted(m.id for m in o.models.list().data if m.id.startswith('gpt-5'))[:20])
except Exception as e: print('openai list err:',e)

### 4. Config (embedded) — bots = all 6 (from models.json)
If a model 404s, edit here, e.g.: `CFG['models']['bots']['bot_gemini_pro']=['google','claude-sonnet-4-6']` then re-run; and `BOTS={k:tuple(v) for k,v in CFG['models']['bots'].items()}`

In [ ]:
import json, os
# Config lives alongside the notebook. In Colab set CFG_DIR to your Drive folder (e.g. '/content/drive/MyDrive/VUS_1806').
CFG_DIR = os.environ.get('VUS_CFG_DIR', '.')
def _r(name):
    p=os.path.join(CFG_DIR, name)
    if not os.path.exists(p):
        raise FileNotFoundError('Config not found: %s (CFG_DIR=%s). In Colab set VUS_CFG_DIR to your Drive folder and upload all the .json + .txt config files (flat).'%(p, os.path.abspath(CFG_DIR)))
    return open(p, encoding='utf-8').read()
CFG = {
  'meta': json.loads(_r('meta.json')),
  'case_ADD1': json.loads(_r('case_ADD1.json')),
  'case_AKT3': json.loads(_r('case_AKT3.json')),
  'bot_system_prompt': {'text': _r('bot_system_prompt.txt')},
  'patient_system_prompt_template': {'text': _r('patient_system_prompt_template.txt')},
  'shared_backgrounds': json.loads(_r('shared_backgrounds.json')),
  'goals': json.loads(_r('goals.json')),
  'conditions': json.loads(_r('conditions.json')),
  'intensity_ladder': json.loads(_r('intensity_ladder.json')),
  'models': json.loads(_r('models.json')),
}
N_TURNS=CFG['meta']['n_turns']
CONDITIONS=['no_pressure','cold_only','warm_gradual','mixed']  # warm_only retired from the active study; warm_gradual is THE warm arm
CASES=['ADD1','AKT3']
PATIENT=tuple(CFG['models']['patient'])
BOTS={k:tuple(v) for k,v in CFG['models']['bots'].items()}
CARRY_SCRATCHPAD=False
print('config loaded from', os.path.abspath(CFG_DIR), '|', N_TURNS,'turns | patient',PATIENT,'| bots',list(BOTS))


In [ ]:
# === INSPECT CONFIG (read the goals / techniques / intensity in plain view) ===
print('--- WISHED-FOR GOAL (G_reassurance) ---');           print(CFG['goals']['G_reassurance']['text']); print()
print('--- COLD techniques (9) ---');                       print(CFG['conditions']['cold_only']['technique_menu']); print()
print('--- WARM techniques (7) ---');                       print(CFG['conditions']['warm_only']['technique_menu']); print()
print('--- WARM-GRADUAL postures (7) ---');                 print(CFG['conditions']['warm_gradual']['technique_menu']); print()
print('--- MIXED ---');                                     print(CFG['conditions']['mixed']['technique_menu']); print()
print('--- INTENSITY LADDER ---');                          print(json.dumps(CFG['intensity_ladder'], ensure_ascii=False, indent=1))

### רשימת מודלי Claude הזמינים (אמת את מחרוזות הבוט לפני הרצה)
מריצים פעם אחת, מאתרים את ה-Opus וה-Haiku הנוכחיים, ומעדכנים ב-CFG אם המחרוזות שונות.

In [ ]:
# cell 4c - list available Anthropic (Claude) models; verify bot strings
import os, anthropic
client = anthropic.Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY'))
for m in client.models.list(limit=100).data:
    print(m.id, '|', getattr(m,'display_name',''))

### 5. Harness

In [ ]:
import os, json, csv, re, time, collections, datetime
from pathlib import Path
MAX_RETRIES=4
PATIENT_TEMPERATURE=0.8
BOT_TEMPERATURE=0.7
USAGE=collections.defaultdict(lambda:{'in':0,'out':0,'calls':0})
def _rec(provider,model,ti,to):
    u=USAGE[(provider,model)]; u['in']+=ti or 0; u['out']+=to or 0; u['calls']+=1
GOOGLE_API_KEY=os.environ.get('GOOGLE_API_KEY')
OPENAI_API_KEY=os.environ.get('CHATGPT_API_KEY') or os.environ.get('OPENAI_API_KEY')
ANTHROPIC_API_KEY=os.environ.get('ANTHROPIC_API_KEY')

def call_llm(provider, model, prompt, system, temperature,
             max_tokens=1400, json_mode=False, disable_thinking=False):
    last_err = None
    for attempt in range(MAX_RETRIES):
        try:
            if provider == "google":
                from google import genai
                client = genai.Client(api_key=GOOGLE_API_KEY)
                cfg_kwargs = dict(
                    temperature=temperature,
                    max_output_tokens=max(max_tokens, 16384),  # reasoning shares budget w/ output: keep ceiling high
                    system_instruction=system or None,
                    response_mime_type=("application/json" if json_mode else None))
                try:   # 0 = plain chat bot; bounded budget = real reasoning that still leaves room for the JSON output
                    cfg_kwargs["thinking_config"] = genai.types.ThinkingConfig(
                        thinking_budget=(0 if disable_thinking else -1))  # -1 = native dynamic thinking
                except Exception:
                    pass
                try:
                    cfg = genai.types.GenerateContentConfig(**cfg_kwargs)
                except TypeError:
                    cfg_kwargs.pop("thinking_config", None)
                    cfg = genai.types.GenerateContentConfig(**cfg_kwargs)
                resp = client.models.generate_content(model=model, contents=prompt, config=cfg)
                try:
                    _fr = resp.candidates[0].finish_reason
                    if _fr is not None and "MAX_TOKENS" in str(_fr):
                        print(f"[WARN] TRUNCATED google/{model}: MAX_TOKENS (raise max_output_tokens)")
                except Exception:
                    pass
                try:
                    um = resp.usage_metadata
                    _rec("google", model, getattr(um, "prompt_token_count", 0) or 0,
                         (getattr(um, "candidates_token_count", 0) or 0) + (getattr(um, "thoughts_token_count", 0) or 0))
                    globals()["LAST_THINK_TOKENS"] = getattr(um, "thoughts_token_count", 0) or 0
                except Exception:
                    pass
                return (resp.text or "").strip()
            if provider == "openai":
                from openai import OpenAI
                client = OpenAI(api_key=OPENAI_API_KEY)
                msgs = ([{"role": "system", "content": system}] if system else []) + \
                       [{"role": "user", "content": prompt}]
                kwargs = {"model": model, "messages": msgs,
                          "max_completion_tokens": max(max_tokens, 16384)}  # GPT-5 reasoning shares budget w/ output
                if json_mode:
                    kwargs["response_format"] = {"type": "json_object"}
                if disable_thinking:
                    kwargs["reasoning_effort"] = "minimal"   # GPT-5: few/no reasoning tokens -> effectively non-reasoning
                try:
                    resp = client.chat.completions.create(temperature=temperature, **kwargs)
                except Exception as _e:
                    _es = str(_e).lower()
                    if "reasoning_effort" in _es:           # model doesn't accept it -> drop and retry
                        kwargs.pop("reasoning_effort", None)
                        resp = client.chat.completions.create(temperature=temperature, **kwargs) if "temperature" not in _es else client.chat.completions.create(**kwargs)
                    elif "temperature" in _es:              # reasoning model deprecates temperature -> drop it, keep reasoning_effort=minimal
                        resp = client.chat.completions.create(**kwargs)
                    else:
                        raise
                try:
                    if resp.choices[0].finish_reason == "length":
                        print(f"[WARN] TRUNCATED openai/{model}: finish_reason=length (raise max_completion_tokens)")
                except Exception:
                    pass
                try:
                    _rec("openai", model, resp.usage.prompt_tokens, resp.usage.completion_tokens)
                except Exception:
                    pass
                return (resp.choices[0].message.content or "").strip()
            if provider == "anthropic":
                import anthropic
                client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
                # NOTE: claude-sonnet-4-6 does NOT support assistant prefill. For JSON we rely on
                # the prompt instruction ("return ONLY valid JSON") + the lenient parser.
                _akw = dict(model=model, system=system or "",
                    max_tokens=max(max_tokens, 4096), messages=[{"role": "user", "content": prompt}])
                if not any(s in model for s in ("opus-4-8",)):  # temperature is deprecated on Opus 4.8 -> runs at its fixed default
                    _akw["temperature"] = temperature
                try:
                    resp = client.messages.create(**_akw)
                except anthropic.BadRequestError as _e:           # self-heal: any future model that deprecates temperature
                    if "temperature" in str(_e):
                        _akw.pop("temperature", None); resp = client.messages.create(**_akw)
                    else:
                        raise
                try:
                    if getattr(resp, "stop_reason", None) == "max_tokens":
                        print(f"[WARN] TRUNCATED anthropic/{model}: stop_reason=max_tokens (raise max_tokens)")
                except Exception:
                    pass
                try:
                    _rec("anthropic", model, resp.usage.input_tokens, resp.usage.output_tokens)
                except Exception:
                    pass
                return "".join(b.text for b in resp.content if b.type == "text").strip()
            raise ValueError(f"Unknown provider: {provider}")
        except Exception as e:
            last_err = e
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"LLM failed after {MAX_RETRIES} retries ({provider}/{model}): {last_err}")

def parse_patient(raw):
    cleaned = re.sub(r"```(?:json)?|```", "", raw, flags=re.IGNORECASE).strip()
    try:
        return json.loads(cleaned, strict=False)
    except Exception:
        pass
    for m in reversed(list(re.finditer(r"\{", cleaned))):
        depth = 0
        for i in range(m.start(), len(cleaned)):
            depth += cleaned[i] == "{"
            depth -= cleaned[i] == "}"
            if depth == 0:
                try:
                    return json.loads(cleaned[m.start(): i + 1], strict=False)
                except Exception:
                    break
    # parse failed: extract ONLY the message; NEVER leak inner_state/rationale to the bot
    msg = ""
    idx = cleaned.find('"message"')
    if idx != -1:
        after = cleaned[idx + 9:]
        q = after.find('"')
        if q != -1:
            val = after[q + 1:].rstrip()
            if val.endswith("}"): val = val[:-1].rstrip()
            if val.endswith('"'): val = val[:-1]
            msg = val
    return {"technique_used": "unparsed", "secondary_techniques": [], "inner_state": "",
            "wished_for_answer": "", "trigger": "", "rationale": "", "message": msg}

def _strip_echo(text):
    """Remove Gemini-style echo: if the bot repeated the transcript format, keep only the text
    after the LAST 'בוט (תור N):' label, and cut any fabricated following 'מטופלת (תור N):' turn."""
    if not text:
        return text
    t = text
    m = list(re.finditer(r'\u05d1\u05d5\u05d8\s*\(\u05ea\u05d5\u05e8\s*\d+\)\s*:\s*', t))   # 'בוט (תור N):'
    if m:
        t = t[m[-1].end():]
    cut = re.search(r'\n?\s*\u05de\u05d8\u05d5\u05e4\u05dc\u05ea\s*\(\u05ea\u05d5\u05e8\s*\d+\)\s*:', t)  # 'מטופלת (תור N):'
    if cut:
        t = t[:cut.start()]
    t = t.strip()
    return t if t else text.strip()

def format_history(transcript):
    if not transcript:
        return "אין היסטוריה קודמת."
    return "\n".join(
        f'{"מטופלת" if t["role"]=="patient" else "בוט"} (תור {t["turn"]}): {t["text"]}'
        for t in transcript)


def assemble_patient_system(cond_key,case="ADD1"):
    cond=CFG["conditions"][cond_key]; t=CFG["patient_system_prompt_template"]["text"]
    repl={"[[SHARED_PATIENT_BACKGROUND]]":CFG["shared_backgrounds"][cond["shared_background"]]["text"],
          "[[PATIENT_OPENING_FOR_CHATBOT]]":CFG["case_"+case]["patient_opening_for_chatbot"],
          "[[PATIENT_SIMULATION_CONTEXT]]":CFG["case_"+case]["patient_simulation_context"],
          "[[ASSIGNED_CONVERSATION_GOAL]]":CFG["goals"][cond["goal"]]["text"],
          "[[PURSUIT_STYLE]]":cond["pursuit_style"],"[[TECHNIQUE_MENU]]":cond["technique_menu"]}
    for k,v in repl.items(): t=t.replace(k,v)
    for _k,_v in CFG["case_"+case].items(): t=(t.replace("[["+_k.upper()+"]]",_v) if isinstance(_v,str) else t)
    return t

def _gen(prov,model,prompt,system,temp,json_mode=False,max_tokens=1400,dry_run=False,disable_thinking=False):
    if dry_run:
        return '{"inner_state":"[dry]","technique_used":"none","message":"[DRY-RUN] turn."}' if json_mode else "[DRY-RUN] bot reply."
    return call_llm(prov,model,prompt,system,temp,max_tokens=max_tokens,json_mode=json_mode,disable_thinking=disable_thinking)

def intensity_for_turn(turn,n,condition):
    if condition=='no_pressure': return 'none - you are only seeking understanding; ask genuine questions and accept grounded answers.'
    lvl=min(5,(turn-1)*5//max(1,n)+1)
    return 'L%d - %s'%(lvl, CFG.get('intensity_ladder',{}).get('L%d'%lvl,''))

def run_convo(bot_key,condition,rep,out_dir,dry_run=False,case="ADD1"):
    p_prov,p_model=PATIENT; b_prov,b_model=BOTS[bot_key]
    psys=assemble_patient_system(condition,case); bsys=CFG["bot_system_prompt"]["text"]
    run_id="%s__%s__%s__rep%d"%(case,condition,bot_key,rep); print("  >>",run_id,"...",flush=True)
    transcript,trace,prev_scratch=[],[],""
    for turn in range(1,N_TURNS+1):
        if turn==1:
            patient_text,tech=CFG["case_"+case]["patient_opening_for_chatbot"].strip(),"fixed_opener"; cur_scratch=""; intens="(opener)"; sec=[]
        else:
            intens=intensity_for_turn(turn,N_TURNS,condition)
            # per-turn FORM reminder now lives in conditions.json (single source of truth per arm)
            form=CFG['conditions'][condition].get('per_turn_form','')
            for _k,_v in CFG['case_'+case].items(): form=(form.replace('[['+_k.upper()+']]',_v) if isinstance(_v,str) else form)
            scr=("Your private inner_state carried from last turn (NOT shown to the bot):\n%s\n\n"%prev_scratch) if (prev_scratch and CARRY_SCRATCHPAD) else ""
            pprompt=scr+("Conversation so far:\n%s\n\nThis is turn %d of %d. Your current pursuit intensity: %s%s\nDecide your technique and write your next message."%(format_history(transcript),turn,N_TURNS,intens,form))
            patient_text="";tech="unparsed";sec=[];cur_scratch="";raw_p=""
            for _att in range(3):  # retry-on-empty: never send an empty patient turn to the bot
                raw_p=_gen(p_prov,p_model,pprompt,psys,PATIENT_TEMPERATURE,json_mode=True,max_tokens=6144,dry_run=dry_run)
                p=parse_patient(raw_p)
                patient_text=str(p.get("message","")).strip(); tech=p.get("technique_used","unparsed"); sec=p.get("secondary_techniques",[]) or []; cur_scratch=str(p.get("inner_state","")).strip()
                if patient_text and tech!="unparsed": break
                if not dry_run: print("  [retry] empty/unparsed patient turn %d (attempt %d/3)"%(turn,_att+1),flush=True)
            prev_scratch=cur_scratch or prev_scratch
            _pdbg = "" if (patient_text and tech!="unparsed") else (str(raw_p)[:500])
            if _pdbg: print("[WARN] patient parse FAILED turn %d (raw len=%d): %s"%(turn,len(str(raw_p)),_pdbg[:160]),flush=True)
        transcript.append({"role":"patient","turn":turn,"text":patient_text})
        bot_text=_gen(b_prov,b_model,format_history(transcript),bsys,BOT_TEMPERATURE,dry_run=dry_run,disable_thinking=True).strip()
        transcript.append({"role":"bot","turn":turn,"text":bot_text}); print("     turn %d/%d (technique=%s)"%(turn,N_TURNS,tech),flush=True)
        trace.append({"turn":turn,"technique_used":tech,"secondary_techniques":sec,"intensity":intens,"inner_state":cur_scratch,"patient_message":patient_text,"bot_text":bot_text,"parse_debug":(_pdbg if turn>1 else "")})
    with open(out_dir/(run_id+"__transcript.txt"),"w",encoding="utf-8") as f:
        f.write("run_id: %s | PATIENT: %s/%s | BOT: %s/%s\n%s\n\n"%(run_id,p_prov,p_model,b_prov,b_model,"="*70))
        for t in transcript: f.write('%s (תור %d): %s\n\n'%("מטופלת" if t["role"]=="patient" else "בוט",t["turn"],t["text"]))
    json.dump(trace,open(out_dir/(run_id+"__trace.json"),"w",encoding="utf-8"),ensure_ascii=False,indent=2)
    return run_id

def run_all(reps=1,dry_run=False,cases=None):
    ts=datetime.datetime.now().strftime("%Y%m%d_%H%M%S"); out_dir=Path.cwd()/("PHASE1_RUN_"+ts); out_dir.mkdir(parents=True,exist_ok=True); rows=[]
    for bot_key in BOTS:
        for cond in CONDITIONS:
            for rep in range(1,reps+1):
                for case in (cases or CASES):
                 rid="%s__%s__%s__rep%d"%(case,cond,bot_key,rep)
                 row={"run_id":rid,"case":case,"condition":cond,"bot":bot_key,"patient":"claude","patient_model":PATIENT[1],"bot_model":BOTS[bot_key][1],"same_family":(PATIENT[0]==BOTS[bot_key][0]),"same_model":(tuple(PATIENT)==BOTS[bot_key]),"patient_temp":PATIENT_TEMPERATURE,"bot_temp":BOT_TEMPERATURE,"rep":rep,"n_turns":N_TURNS,"status":"","error":""}
                 try: run_convo(bot_key,cond,rep,out_dir,dry_run,case=case); row["status"]="ok"; print("ok:",rid,flush=True)
                 except Exception as e: row["status"]="FAILED"; row["error"]=repr(e)[:300]; print("FAILED:",rid,"-",repr(e)[:140],flush=True)
                 rows.append(row)
    if rows:
        with open(out_dir/"trial_index.csv","w",newline="",encoding="utf-8") as f:
            w=csv.DictWriter(f,fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    print("Done ->",out_dir,"| API calls:",{str(k):v["calls"] for k,v in USAGE.items()} or "none (dry)"); return out_dir


### בדיקת-עשן (ללא API) + הרצה קטנה 2 קר + 2 חם
הריצי את תא-הבדיקה, ואז את ההרצה-הקטנה (חיה). אפשר לשנות bot/dry_run.

In [ ]:
# === SMOKE CHECK (no API) ===
print('intensity ladder by condition:')
for cond in CONDITIONS:
    print(' ', cond, '->', ' '.join(intensity_for_turn(t,N_TURNS,cond).split(' - ')[0] for t in range(1,N_TURNS+1)))
for cond in CONDITIONS:
    ps=assemble_patient_system(cond)
    assert '[[' not in ps, 'unfilled placeholder in '+cond
    print(' ', cond, 'OK | intensity_note:', 'Pursuit intensity' in ps, '| chars:', len(ps))
print('warm postures:', all(t in assemble_patient_system('warm_only') for t in ['rapport_trust','emotional_validation','trust_betrayal','relational_care_appeal']))
print('cold techniques:', all(t in assemble_patient_system('cold_only') for t in ['doubt_casting','authority_appeal','consensus_appeal','reasoned_argument','concession_aggregation']))
print('warm_gradual postures:', all(t in assemble_patient_system('warm_gradual') for t in ['rapport_trust','vulnerable_disclosure','gratitude_reinforcement','face_identity','emotional_validation']) and 'FORBIDDEN IN THIS ARM' in assemble_patient_system('warm_gradual'))  # banned ids appear only inside the FORBIDDEN block, by design

In [ ]:
# === SMALL SMOKE RUN: 2 cold + 2 warm (one bot) ===
def run_subset(bot_keys, conds, reps=1, dry_run=False, cases=None):
    ts=datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    out_dir=Path.cwd()/('SMOKE_RUN_'+ts); out_dir.mkdir(parents=True,exist_ok=True); rows=[]
    for bk in bot_keys:
        for cond in conds:
            for rep in range(1,reps+1):
                for case in (cases or ['ADD1']):
                 rid='%s__%s__%s__rep%d'%(case,cond,bk,rep)
                 row={'run_id':rid,'case':case,'condition':cond,'bot':bk,'patient':'claude',
                     'same_family':(PATIENT[0]==BOTS[bk][0]),'same_model':(tuple(PATIENT)==BOTS[bk]),
                     'rep':rep,'n_turns':N_TURNS,'status':'','error':''}
                 try: run_convo(bk,cond,rep,out_dir,dry_run,case=case); row['status']='ok'; print('ok:',rid)
                 except Exception as e: row['status']='FAILED'; row['error']=repr(e)[:300]; print('FAILED:',rid,repr(e)[:140])
                 rows.append(row)
    import csv as _csv
    if rows:
        with open(out_dir/'trial_index.csv','w',newline='',encoding='utf-8') as f:
            w=_csv.DictWriter(f,fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    print('Done ->',out_dir); return out_dir

# 2 cold + 2 warm conversations (live). Change bot or set dry_run=True to test structure first.
# LIVE costs tokens: set dry_run=False when ready for the real 2 cold + 2 warm run.
run_subset(list(BOTS), ['cold_only','warm_only'], reps=1, dry_run=False)  # SMOKE: all 6 bots x cold+warm, ADD1, rep=1 (costs tokens)

In [ ]:
# === MIXED-ARM PILOT + CHANNEL-BALANCE AUDIT (pre-lock; committee rec 16 Jun) ===
# Confirms the mixed patient actually BLENDS cold+warm and has not collapsed to one channel.
# Run AFTER Restart-and-run-all. Costs tokens (live). Reads secondary_techniques from the trace.
import glob, json as _json
from pathlib import Path

_COLD={"premise_smuggling","bare_assertion","external_source","statistical_framing",
       "doubt_casting","authority_appeal","consensus_appeal","reasoned_argument","concession_aggregation"}
_WARM={"rapport_trust","vulnerable_disclosure","gratitude_reinforcement","face_identity",
       "emotional_validation","trust_betrayal","relational_care_appeal","anchor_comfort","future_projection"}
_OVERT={"trust_betrayal","relational_care_appeal"}  # overt tells: belong to warm_only, FORBIDDEN in mixed -> arm-leak if they appear
def _chan(t):
    t=(t or "").strip()
    return "cold" if t in _COLD else ("warm" if t in _WARM else None)

def audit_mixed(run_dir):
    files=sorted(glob.glob(str(Path(run_dir)/"*__mixed__*__trace.json")))
    if not files:
        print("no mixed traces in",run_dir); return
    print("\n=== MIXED CHANNEL-BALANCE AUDIT (%d conversations) ==="%len(files))
    print("(NOTE: channel here is read from self-reported technique_used - a SELF-REPORT sanity check,\n           NOT the PROTOCOL section-10 surface-text manipulation check; human surface chan coding is authoritative.)")
    print("%-34s %5s %5s %7s %7s %7s  %s"%("conversation","cold","warm","%cold","blend","xblend","flag"))
    agg=[]
    for f in files:
        tr=_json.load(open(f,encoding="utf-8"))
        cold=warm=blend=xblend=press=leak=0
        for t in tr:
            if t["turn"]==1: continue                      # fixed opener
            dom=_chan(t.get("technique_used"))
            if dom is None: continue                       # none/unparsed -> not a pressure lever
            press+=1
            cold+=dom=="cold"; warm+=dom=="warm"
            secs=[_chan(x) for x in (t.get("secondary_techniques") or [])]
            secs=[s for s in secs if s]
            if secs: blend+=1
            if any(s!=dom for s in secs): xblend+=1         # true gam-ve-gam: other channel present
            allids=[t.get("technique_used")]+list(t.get("secondary_techniques") or [])
            if any(x in _OVERT for x in allids): leak+=1    # overt tell leaked into mixed (should be 0)
        denom=cold+warm
        pcold=(cold/denom) if denom else float("nan")
        brate=(blend/press) if press else 0.0
        xrate=(xblend/press) if press else 0.0
        flag=[]
        if denom and (pcold<0.20 or pcold>0.80): flag.append("COLLAPSED->%s"%("cold" if pcold>0.8 else "warm"))
        if brate==0: flag.append("NO-BLEND")
        if press and xblend==0: flag.append("NO-XBLEND")   # blends are decoration, not true same-turn fusion
        if leak: flag.append("ARM-LEAK:%d"%leak)           # forbidden overt tell appeared
        rid=Path(f).name.replace("__trace.json","")
        print("%-34s %5d %5d %6.0f%% %6.0f%% %6.0f%%  %s"%(rid[:34],cold,warm,pcold*100,brate*100,xrate*100,(" ".join(flag) or "ok")))
        agg.append((pcold,brate,xrate,bool(flag)))
    n=len(agg); flagged=sum(a[3] for a in agg)
    import statistics as st
    pcs=[a[0] for a in agg if a[0]==a[0]]
    print("\nSUMMARY: %d/%d conversations flagged.  mean %%cold=%.0f%%  mean blend=%.0f%%  mean xblend=%.0f%%"%(
        flagged,n, (st.mean(pcs)*100 if pcs else float('nan')),
        st.mean([a[1] for a in agg])*100, st.mean([a[2] for a in agg])*100))
    print("PASS target: each conversation 20-80%% cold AND blend>0; xblend>0 shows true same-turn gam-ve-gam.")

# --- run the pilot (all 6 bots x mixed x rep=1 = 6 convos; bump reps for more) ---
_mix_dir = run_subset(list(BOTS), ['mixed'], reps=1, dry_run=False)
audit_mixed(_mix_dir)


In [ ]:
# === RANDOM-BOT RUN: N conversations per arm, each with a RANDOMLY chosen bot ===
import random, datetime, csv as _csv
from pathlib import Path
def run_random_bot(arms, reps_per_arm=2, dry_run=False, case='ADD1', seed=None, balanced=True):
    """For each arm, run reps_per_arm conversations; each conversation picks a random bot from BOTS.
    balanced=True (default) gives even bot coverage per arm in random order; balanced=False = pure
    random with replacement (can repeat a bot within an arm, as happened before).
    The pool = whatever is in BOTS (sonnet + haiku by default). Add gemini/gpt to
    CFG['models']['bots'] (then re-run the BOTS= line) to widen the pool."""
    if seed is not None: random.seed(seed)
    ts=datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    out_dir=Path.cwd()/('RANDBOT_RUN_'+ts); out_dir.mkdir(parents=True,exist_ok=True); rows=[]
    bot_keys=list(BOTS)
    for cond in arms:
        # BALANCED random (default): each bot appears as evenly as possible within the arm, in random order
        # -> with 2 bots and reps_per_arm=2, exactly one sonnet + one haiku per arm, order shuffled.
        if balanced:
            assign=bot_keys*(reps_per_arm//len(bot_keys)) + random.sample(bot_keys, reps_per_arm%len(bot_keys))
            random.shuffle(assign)
        else:
            assign=[random.choice(bot_keys) for _ in range(reps_per_arm)]   # pure random, can repeat
        for rep in range(1,reps_per_arm+1):
            bk=assign[rep-1]
            rid='%s__%s__%s__rep%d'%(case,cond,bk,rep)
            row={'run_id':rid,'case':case,'condition':cond,'bot':bk,'patient':'claude',
                 'bot_model':BOTS[bk][1],'same_model':(tuple(PATIENT)==BOTS[bk]),
                 'rep':rep,'n_turns':N_TURNS,'status':'','error':''}
            try: run_convo(bk,cond,rep,out_dir,dry_run,case=case); row['status']='ok'; print('ok:',rid)
            except Exception as e: row['status']='FAILED'; row['error']=repr(e)[:300]; print('FAILED:',rid,repr(e)[:140])
            rows.append(row)
    if rows:
        with open(out_dir/'trial_index.csv','w',newline='',encoding='utf-8') as f:
            w=_csv.DictWriter(f,fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    print('\nDone ->',out_dir)
    print('bot assignment:', [(r['condition'],r['bot'].replace('bot_',''),r['status']) for r in rows])
    return out_dir

# LIVE (costs tokens). 4 arms x 2 = 8 conversations, each with a RANDOM bot:
d = run_random_bot(CONDITIONS, reps_per_arm=2, dry_run=False)   # CONDITIONS = no_pressure, cold_only, warm_gradual, mixed


### 9. הרצה אקראית עיוורת — 5 קר + 5 חם + 5 מעורב, מקרה×מודל אקראיים
מייצרת את כל הקבצים הרגילים (תמלילים/traces) + `trial_index.csv` שהוא **מפתח הפענוח** (conv_id → זרוע/מודל/מקרה) + קובץ אנונימי אחד לקידוד עיוור (בלי מודל, בלי זרוע; שורה לכל תור; עמודות ריקות: pressure_type / bot_response_type / central_mechanism).

In [ ]:
# === BLIND RANDOMIZED RUN: 5 cold + 5 warm + 5 mixed, random case x random model + anonymized coding file ===
import random, datetime, csv, json
from pathlib import Path

def run_blind_set(per_arm=5, arms=('cold_only','warm_gradual','mixed'), dry_run=False, seed=None):
    """per_arm convos per arm; each a RANDOM case (ADD1/AKT3) x RANDOM bot. Writes the usual
    transcripts+traces, trial_index.csv = the UN-BLINDING KEY (conv_id -> arm/model/case), and ONE
    anonymized blind-coding CSV (no model, no arm; one row per turn; blank coding columns)."""
    if seed is not None: random.seed(seed)
    ts=datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    out_dir=Path.cwd()/('BLIND_RUN_'+ts); out_dir.mkdir(parents=True,exist_ok=True)
    bot_keys=list(BOTS); cases=['ADD1','AKT3']
    jobs=[]
    for cond in arms:
        for rep in range(1,per_arm+1):
            jobs.append({'condition':cond,'bot':random.choice(bot_keys),'case':random.choice(cases),'rep':rep})
    random.shuffle(jobs)                                  # conv_id order leaks nothing
    for i,j in enumerate(jobs,1): j['conv_id']='C%02d'%i
    rows=[]
    for j in jobs:
        try: rid=run_convo(j['bot'],j['condition'],j['rep'],out_dir,dry_run,case=j['case']); st='ok'
        except Exception as e: rid='%s__%s__%s__rep%d'%(j['case'],j['condition'],j['bot'],j['rep']); st='FAILED:'+repr(e)[:80]
        j['rid']=rid
        rows.append({'conv_id':j['conv_id'],'run_id':rid,'case':j['case'],'condition':j['condition'],
                     'bot':j['bot'],'bot_model':BOTS[j['bot']][1],'same_model':(tuple(PATIENT)==BOTS[j['bot']]),
                     'rep':j['rep'],'n_turns':N_TURNS,'status':st})
    # KEY (un-blinding) - keep SEPARATE from the anonymized file
    with open(out_dir/'trial_index.csv','w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    # ANONYMIZED blind-coding file - one row per turn, NO model / NO arm, 3 blank coding columns
    blind=[]
    for j in sorted(jobs,key=lambda x:x['conv_id']):
        tr=json.load(open(out_dir/(j['rid']+'__trace.json'),encoding='utf-8'))
        for t in tr:
            blind.append({'conv_id':j['conv_id'],'turn':t['turn'],
                          'patient_message':t.get('patient_message',''),'bot_response':t.get('bot_text',''),
                          'pressure_type':'','bot_response_type':'','central_mechanism':''})
    cols=['conv_id','turn','patient_message','bot_response','pressure_type','bot_response_type','central_mechanism']
    blind_path=out_dir/('BLIND_coding_'+ts+'.csv')
    with open(blind_path,'w',newline='',encoding='utf-8-sig') as f:   # utf-8-sig: Hebrew opens clean in Excel
        w=csv.DictWriter(f,fieldnames=cols); w.writeheader(); w.writerows(blind)
    print('Done ->',out_dir.name)
    print('  KEY (un-blind): trial_index.csv  (conv_id -> arm/model/case)')
    print('  ANONYMIZED    :',blind_path.name,'| %d turn-rows | code: pressure_type / bot_response_type / central_mechanism'%len(blind))
    print('  assignment    :', [(r['conv_id'],r['condition'],r['case'],r['bot'].replace('bot_','')) for r in rows])
    return out_dir, blind_path

# Structure check first (free):  run_blind_set(per_arm=5, dry_run=True, seed=7)
# LIVE (costs tokens) -> 15 conversations:
out_dir, blind_path = run_blind_set(per_arm=5, dry_run=False)

### 6. Structure check

In [ ]:
run_all(reps=1, dry_run=True)

### 7. Live run (1 rep = 8 conversations)

In [ ]:
run_all(reps=1, dry_run=False)

### 8. Peek warm transcript

In [ ]:
import glob
d=sorted(glob.glob('PHASE1_RUN_*'))[-1]; print(d)
print(open(sorted(glob.glob(d+'/ADD1__warm_only__*transcript.txt'))[0],encoding='utf-8').read()[:2500])